# Campagne CipherMark / WAF — Colab T4

Exécuter les cellules dans l'ordre. Les résultats sont sauvegardés dans Google Drive.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CODE_DIR='/content/drive/MyDrive/code-memoire'
if not os.path.isdir(CODE_DIR): raise FileNotFoundError(f'Dépôt introuvable: {CODE_DIR}')
os.chdir(CODE_DIR)
print(os.getcwd())

In [ ]:
!python -m pip install -q -r requirements.txt 2>/dev/null || true
!python -m pip install -q lpips torchmetrics einops omegaconf safetensors 2>/dev/null || true
!python -m py_compile scripts/ciphermark/eval_grande_echelle_decodeur.py
print('Dépendances et syntaxe vérifiées.')

## Smoke test (à exécuter en premier)

In [ ]:
import os
os.environ['N']='100'; os.environ['N_GEN']='20'; os.environ['WITH_LPIPS']='0'
os.environ['SORTIE']='/content/drive/MyDrive/ciphermark_runs/smoke'
!bash runpod/campagne_metriques_unifiees.sh

## Campagne complète

Lancer seulement après un smoke test réussi. Les fichiers existants sont ignorés pour permettre la reprise.

In [ ]:
import os
os.environ['N']='5000'; os.environ['N_GEN']='1000'; os.environ['WITH_LPIPS']='1'
os.environ['SORTIE']='/content/drive/MyDrive/ciphermark_runs/final'
!bash runpod/campagne_metriques_unifiees.sh

In [ ]:
from pathlib import Path
root=Path('/content/drive/MyDrive/ciphermark_runs')
for p in sorted(root.rglob('*.json')): print(p, p.stat().st_size//1024, 'Ko')